# AI Location Embedding Recommender

This notebook builds embeddings and a recommendation workflow for the final AI-enriched cafe/bar/location dataset.

Important design choices:
- Embedding text uses AI summaries, tags, axes, amenities, useful types, and price discovery.
- Raw reviews are not embedded again because their signal is already compressed into the AI summary fields.
- Ratings, review counts, and map visibility are used for filtering/ranking, not as the main semantic text.
- Batch submission is guarded by `MANUAL_SUBMIT = False` so Run All will not accidentally submit paid API jobs.

In [2]:
from pathlib import Path
import os
import sys
import json
import math
import importlib

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", None)


def find_project_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists():
            return candidate
    raise FileNotFoundError("Could not find project root with .env")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from recommendation_system.ai_location_recommender import location_recommender_utils as rec
rec = importlib.reload(rec)

print("Project root:", PROJECT_ROOT)
print("Source CSV:", rec.SOURCE_CSV_PATH)
print("Recommender root:", rec.RECOMMENDER_ROOT)

Project root: /Users/ilya/Documents/VisualStudioCode/SLOCO
Source CSV: /Users/ilya/Documents/VisualStudioCode/SLOCO/data_scraping/output/ai_location_summaries/final/final_ai_dataframe_with_map_scores_latest.csv
Recommender root: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender


In [3]:
# Configuration
EMBEDDING_MODEL = rec.DEFAULT_EMBEDDING_MODEL  # Change to "text-embedding-3-large" for a comparison run.
MAX_INPUTS_PER_BATCH = rec.MAX_EMBEDDING_INPUTS_PER_BATCH
SOURCE_CSV_PATH = rec.SOURCE_CSV_PATH

artifact_dirs = rec.ensure_artifact_dirs()
for name, path in artifact_dirs.items():
    print(f"{name}: {path}")

print("Embedding model:", EMBEDDING_MODEL)

data: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data
embedding_requests: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/embedding_requests
batch_manifests: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/batch_manifests
batch_outputs: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/batch_outputs
embedding_store: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/embedding_store
exports: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/exports
Embedding model: text-embedding-3-small


## 1. Load And Inspect Final Dataset

In [4]:
df_raw = rec.load_source_dataframe(SOURCE_CSV_PATH)

print("Rows:", len(df_raw))
print("Columns:", len(df_raw.columns))
display(df_raw.head(3))

selected_input_columns = [
    "name", "primary_type", "types", "serves", "features",
    "ai_card_summary", "ai_place_type_summary", "ai_vibe", "ai_what_to_expect",
    "ai_food_and_drinks", "ai_price", "ai_service", "ai_the_move", "ai_watch_out",
    "ai_tags_csv", "ai_tags_json",
    "axis_quiet_lively", "axis_work_social", "axis_day_night", "axis_casual_premium",
    "axis_drinks_food", "axis_local_tourist", "axis_cheap_expensive", "axis_traditional_experimental",
    "price_level", "price_min_ron", "price_max_ron",
]
ranking_only_columns = [
    "place_id", "google_rating", "google_user_rating_count", "apify_review_count",
    "apify_rating_avg", "review_rating_distribution", "ai_confidence",
    "map_visibility_score", "map_min_zoom_global",
]

print("Missing share for embedding input columns:")
display(df_raw[selected_input_columns].isna().mean().sort_values(ascending=False).to_frame("missing_share"))

print("Missing share for ranking/matching columns:")
display(df_raw[ranking_only_columns].isna().mean().sort_values(ascending=False).to_frame("missing_share"))

Rows: 2508
Columns: 55


place_id                      name primary_type  \
0  ChIJaX6OBQD_sUARpsOX3IMt2R4       "Latte" Coffe to go  coffee_shop   
1  ChIJ_xaqNQMCskAR8aQ8oHos1Ro         "Seneca Anticafe"   book_store   
2  ChIJnUA9FgD_sUAR2ooJ3CWb0Zc  "TABERNA" LA ULTIMA Y YA          bar   

                                                                                                                           types  \
0                                   ['coffee_shop', 'cafe', 'food_store', 'food', 'point_of_interest', 'store', 'establishment']   
1  ['book_store', 'coffee_shop', 'cafe', 'coworking_space', 'point_of_interest', 'food_store', 'food', 'store', 'establishment']   
2                                                                                  ['bar', 'point_of_interest', 'establishment']   

   google_rating  google_user_rating_count  apify_review_count  \
0            1.0                       1.0                   1   
1            4.8                    1411.0                  50   
2            NaN                       NaN                   0   

   apify_rating_avg         review_rating_distribution price_level  \
0               1.0                           {'1': 1}         NaN   
1               4.8  {'1': 1, '3': 2, '4': 2, '5': 45}         NaN   
2               NaN                                 {}         NaN   

   price_min_ron  price_max_ron            serves  \
0            NaN            NaN        ['coffee']   
1            NaN            NaN                []   
2            NaN            NaN  ['beer', 'wine']   

                                                    features  has_ai_summary  \
0                       {'dineIn': True, 'liveMusic': False}            True   
1                                         {'delivery': True}            True   
2  {'dineIn': True, 'goodForGroups': True, 'restroom': True}            True   

                                                                                                                                ai_card_summary  \
0                 A bare-bones coffee stop for a cheap espresso in a pinch, but the coffee is weak and easy to skip for anything better nearby.   
1  A cozy bookshop-coworking café with quiet workspaces, tea and snacks included, and a calm, home-like atmosphere for reading or focused work.   
2                                     A laid-back neighborhood bar for beer and wine, with a simple, social feel and an easygoing stop-in pace.   

      ai_place_type_summary  \
0              coffee stand   
1  bookstore coworking cafe   
2          neighborhood bar   

                                                                                                                                                                                                                     ai_vibe  \
0                                                                              It feels bare-bones and utilitarian, more like a quick machine stop than a true cafe. The setup is plain and functional rather than inviting.   
1  It feels calm, cozy, and book-lined, with a home-like atmosphere that suits reading, studying, and long work sessions. The space also hosts events and workshops, so it can shift from quiet focus to community activity.   
2                   It feels casual and unpretentious, with a straightforward bar atmosphere built around drinks and conversation. The space reads as an easygoing spot for a relaxed stop rather than a polished night out.   

                                                                                                                                                                                                  ai_what_to_expect  \
0                                                                               Expect a very simple grab-and-go stop with little comfort or linger time. It fits a fast caffeine fix, not a sit-down coffee break.   
1  Expect a self-serve, time-based space with tables, sofas, and quieter rooms 

Missing share for embedding input columns:


,missing_share
price_level,0.743222
price_max_ron,0.389553
price_min_ron,0.379984
ai_watch_out,0.065789
primary_type,0.002392
ai_service,0.000399
features,0.000000
axis_work_social,0.000000
types,0.000000
serves,0.000000


Missing share for ranking/matching columns:


,missing_share
google_rating,0.100478
google_user_rating_count,0.100478
apify_rating_avg,0.100478
place_id,0.000000
apify_review_count,0.000000
review_rating_distribution,0.000000
ai_confidence,0.000000
map_visibility_score,0.000000
map_min_zoom_global,0.000000


## 2. Prepare Embedding Text

The prepared text deliberately excludes technical IDs, model metadata, token usage, and raw reviews.

In [5]:
df_prepared = rec.add_embedding_preparation_columns(df_raw)

print("Prepared shape:", df_prepared.shape)
print("Duplicate place_id:", int(df_prepared["place_id"].duplicated().sum()))
print("Empty embedding_text:", int((df_prepared["embedding_text"].str.len() == 0).sum()))

preview_cols = [
    "source_row_index", "place_id", "name", "primary_type", "ai_place_type_summary",
    "semantic_info_score", "price_bucket", "embedding_text_hash", "ai_confidence", "map_visibility_score",
]
display(df_prepared[preview_cols].head(10))

print("Embedding text length stats:")
display(df_prepared["embedding_text"].str.len().describe(percentiles=[0.1, 0.5, 0.9, 0.95, 0.99]).to_frame("chars"))

Prepared shape: (2508, 65)
Duplicate place_id: 0
Empty embedding_text: 0


,source_row_index,place_id,name,primary_type,ai_place_type_summary,semantic_info_score,price_bucket,embedding_text_hash,ai_confidence,map_visibility_score
0,0,ChIJaX6OBQD_sUARpsOX3IMt2R4,"""Latte"" Coffe to go",coffee_shop,coffee stand,15.57,budget,e86e286ebaca328affe102cf956b4b9e1cf08c6a8e74bef0bcdeb3dac3b2bf02,high,18.9
1,1,ChIJ_xaqNQMCskAR8aQ8oHos1Ro,"""Seneca Anticafe""",book_store,bookstore coworking cafe,19.05,moderate,850254ad55dc1e94db644e82524bed12684fe8389e36c9f626956ad20b8fe8b4,high,89.9
2,2,ChIJnUA9FgD_sUAR2ooJ3CWb0Zc,"""TABERNA"" LA ULTIMA Y YA",bar,neighborhood bar,13.10,budget,1a03c43358d8e8b754b69d7f5501c15f69b551625ec8a8ff120c33eea4a9900e,low,5.0
3,3,ChIJAQAckz__sUARPQnYMKtxIRQ,#ALTFEL,lounge_bar,lounge bar,21.20,"moderate, observed range 20-100 RON",059da87008f78ebbd452de6b4bf2bf4eb6ee96fa83869f49275cdd498cac9087,high,65.1
4,4,ChIJbYXdJwD_sUAR2yjbAGOKJPM,#Altfel Park Lake,bar,terrace bar,17.34,"moderate, observed range 20-40 RON",dc81af8a188a974b05c637645ae8577546eba6121daa8f036661b548f000ae18,medium,33.1
5,5,ChIJrRz_N5j_sUAReHHbh0ZcBlw,#aici,lounge_bar,riverfront lounge bar,20.45,"moderate, observed range 40-140 RON",1172b5c8739b9b187c109ba463ea810615b63d1b82e3b1629b29bf5c29cb7f9a,high,64.1
6,6,ChIJt2RQCJ7_sUARWXvXE7IIORg,#altfel Unirii,lounge_bar,lounge bar,20.65,"moderate, observed range 20-40 RON",4f9149edebc4e030f333b06cba6e850725e2d635ab74f494cbba178906f72474,high,55.8
7,7,ChIJTcMzKgD_sUARRP16DLrUKFU,.,bar,bar,13.45,moderate,4311735a1c7813fbeaee2d8e7629eee615982d72635b0b134fdc1563e7710fff,low,5.0
8,8,ChIJYRw_Y83_sUARMdUkJ_zovRw,.,bar,park terrace bar,18.75,"budget, observed range 20-40 RON",a4ddc93692e4918c248bbf5303aa4f17c511a1f0d48a739a0f732b5fbcf133de,high,60.2
9,9,ChIJi4adDnsCskARKZC58K1UUqg,1 Minute Floreasca Park,convenience_store,convenience store cafe,15.59,premium,8beccf56ff260ecb0c9195864ee7217db7659f0015575ddddd628976a3661eca,medium,55.4


Embedding text length stats:


,chars
count,2508.000000
mean,2267.426236
std,309.011919
min,1208.000000
10%,1826.700000
50%,2297.000000
90%,2654.000000
95%,2742.000000
99%,2873.860000
max,3059.000000


In [6]:
df_prepared

place_id                           name primary_type  \
0     ChIJaX6OBQD_sUARpsOX3IMt2R4            "Latte" Coffe to go  coffee_shop   
1     ChIJ_xaqNQMCskAR8aQ8oHos1Ro              "Seneca Anticafe"   book_store   
2     ChIJnUA9FgD_sUAR2ooJ3CWb0Zc       "TABERNA" LA ULTIMA Y YA          bar   
3     ChIJAQAckz__sUARPQnYMKtxIRQ                        #ALTFEL   lounge_bar   
4     ChIJbYXdJwD_sUAR2yjbAGOKJPM              #Altfel Park Lake          bar   
...                           ...                            ...          ...   
2503  ChIJYRGVLgD_sUARJXu-GmrIc3k                         Μαγαζί          bar   
2504  ChIJud_K0Mf_sUARCBOJ3CqSGy4                            Їжа          bar   
2505  ChIJKWe2SAD_sUARyR2nztHkGXI               בית קפה מרה מורה  coffee_shop   
2506  ChIJlVw_XWP_sUARSKlubPOvqZY  “ Caffe' del Moro “ Bucharest         cafe   
2507  ChIJAe7KFAD_sUARLcNGiAmXP-g           🌈Rainbow Coffee&more          bar   

                                                                                                                              types  \
0                                      ['coffee_shop', 'cafe', 'food_store', 'food', 'point_of_interest', 'store', 'establishment']   
1     ['book_store', 'coffee_shop', 'cafe', 'coworking_space', 'point_of_interest', 'food_store', 'food', 'store', 'establishment']   
2                                                                                     ['bar', 'point_of_interest', 'establishment']   
3                          ['lounge_bar', 'night_club', 'bar', 'point_of_interest', 'association_or_organization', 'establishment']   
4                                                                                     ['bar', 'point_of_interest', 'establishment']   
...                                                                                                                             ...   
2503                                                                                  ['bar', 'point_of_interest', 'establishment']   
2504                                                                                  ['bar', 'point_of_interest', 'establishment']   
2505                                   ['coffee_shop', 'cafe', 'food_store', 'food', 'store', 'point_of_interest', 'establishment']   
2506         ['cafe', 'bistro', 'grocery_store', 'food_store', 'restaurant', 'food', 'point_of_interest', 'store', 'establishment']   
2507                                                                                  ['bar', 'point_of_interest', 'establishment']   

      google_rating  google_user_rating_count  apify_review_count  \
0               1.0                       1.0                   1   
1               4.8                    1411.0                  50   
2               NaN                       NaN                   0   
3               4.2                     909.0                  50   
4               3.7                      12.0                  12   
...             ...                       ...                 ...   
2503            5.0                       1.0                   1   
2504            5.0                       1.0                   1   
2505            NaN                       NaN                   0   
2506            4.2                     316.0                  50   
2507            4.7                      61.0                  50   

      apify_rating_avg                 review_rating_distribution  \
0               1.0000                                   {'1': 1}   
1               4.8000          {'1': 1, '3': 2, '4': 2, '5': 45}   
2                  NaN                                         {}   
3               3.9200  {'1': 5, '2': 5, '3': 5, '4': 9, '5': 26}   
4               3.6667                           {'1': 4, '5': 8}   
...                ...                                        ...   
2503            5.0000                                   {'5': 1}   
2504            5.0000                            

In [7]:
# Full, untruncated input preview. Change sample_indices to inspect specific rows.
sample_indices = [0, 1, 3]

for idx in sample_indices:
    row = df_prepared.loc[idx]
    print("=" * 120)
    print(f"ROW {idx}: {row['name']} | place_id={row['place_id']}")
    print("=" * 120)
    print(row["embedding_text"])
    print()

ROW 0: "Latte" Coffe to go | place_id=ChIJaX6OBQD_sUARpsOX3IMt2R4
Venue name: "Latte" Coffe to go
Primary type: coffee_shop
AI place type: coffee stand
Useful Google types: coffee_shop, cafe
Core description: A bare-bones coffee stop for a cheap espresso in a pinch, but the coffee is weak and easy to skip for anything better nearby.
Vibe and atmosphere: It feels bare-bones and utilitarian, more like a quick machine stop than a true cafe. The setup is plain and functional rather than inviting.
What to expect: Expect a very simple grab-and-go stop with little comfort or linger time. It fits a fast caffeine fix, not a sit-down coffee break.
Food and drinks: Coffee is the only clear offering, and the espresso is extremely basic. It is a drinks-first stop with no sense of quality beyond convenience.
Price and value: It is very cheap, around the low end for espresso. The value is weak unless speed matters more than taste. budget
Service: Service is minimal and transactional, as you would exp

## 3. Token And Cost Estimate

In [8]:
token_summary = rec.summarize_embedding_token_usage(
    df_prepared,
    text_col="embedding_text",
    model=EMBEDDING_MODEL,
    max_inputs_per_batch=MAX_INPUTS_PER_BATCH,
)
token_summary

{'rows': 2508,
 'total_tokens': 1287392,
 'avg_tokens_per_row': 513.3141945773525,
 'max_tokens_per_row': 684,
 'estimated_batch_cost_usd': 0.01287392,
 'model': 'text-embedding-3-small',
 'max_inputs_per_batch': 10000,
 'estimated_batches': 1}

## 4. Dry Run Batch JSONL

This writes only the first 10 requests locally and validates the JSONL structure. It does not call the OpenAI API.

In [9]:
dry_run_df = df_prepared.head(10).copy()
dry_run_path = rec.EMBEDDING_REQUEST_DIR / "dry_run_10_location_embedding_requests.jsonl"
rec.write_embedding_batch_jsonl(dry_run_df, dry_run_path, text_col="embedding_text", model=EMBEDDING_MODEL)
dry_run_validation = rec.validate_embedding_batch_jsonl(dry_run_path)
dry_run_validation

{'path': '/Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/embedding_requests/dry_run_10_location_embedding_requests.jsonl',
 'line_count': 10,
 'unique_custom_ids': 10,
 'duplicate_custom_ids': 0,
 'models': ['text-embedding-3-small'],
 'urls': ['/v1/embeddings'],
 'size_mb': 0.023220062255859375}

## 5. Submit Embedding Batch

This cell is intentionally guarded. Set `MANUAL_SUBMIT = True` only when you are ready to send real Batch API jobs.

In [10]:
MANUAL_SUBMIT = True

if MANUAL_SUBMIT:
    submitted_embedding_batches = rec.submit_embedding_batches(
        df_prepared,
        text_col="embedding_text",
        model=EMBEDDING_MODEL,
        max_inputs_per_batch=MAX_INPUTS_PER_BATCH,
        source_csv_path=SOURCE_CSV_PATH,
    )
    display(pd.DataFrame(submitted_embedding_batches))
else:
    print("MANUAL_SUBMIT is False. Set it to True to submit real OpenAI Batch API jobs.")
    print("Manifest path:", rec.BATCH_MANIFEST_PATH)

Submitted batch 1: batch_6a1c7220819c81908706c380752a4ce9 (0:2508, 2508 inputs)


,run_id,batch_number,batch_id,input_file_id,jsonl_path,row_start,row_end,n_inputs,estimated_input_tokens,estimated_batch_cost_usd,model,dimensions,endpoint,source_csv_path,submitted_at_utc
0,20260531T173837Z,1,batch_6a1c7220819c81908706c380752a4ce9,file-32fhn4W6VBPUCbqafwDifG,/Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/embedding_requests/location_embedding_requests_20260531T173837Z_part_001_0_2508.jsonl,0,2508,2508,1287392,0.012874,text-embedding-3-small,None,/v1/embeddings,/Users/ilya/Documents/VisualStudioCode/SLOCO/data_scraping/output/ai_location_summaries/final/final_ai_dataframe_with_map_scores_latest.csv,2026-05-31T17:38:41.322971+00:00


## 6. Check Batch Status

Run this whenever you want to refresh Batch API status.

In [10]:
# This calls the OpenAI API if a manifest exists.
try:
    batch_status_df = rec.check_embedding_batch_statuses()
    display(batch_status_df)
    if not batch_status_df.empty:
        completed = batch_status_df["completed"].fillna(0).sum()
        total = batch_status_df["total"].fillna(0).sum()
        print(f"Completed requests: {completed:g} / {total:g}")
except Exception as exc:
    print(type(exc).__name__, str(exc))

,run_id,batch_number,batch_id,status,total,completed,failed,output_file_id,error_file_id,n_inputs,estimated_input_tokens,estimated_batch_cost_usd,submitted_at_utc
0,20260531T173837Z,1,batch_6a1c7220819c81908706c380752a4ce9,completed,2508,2508,0,file-2mijAtxx8dXZs6STUUv7TA,None,2508,1287392,0.012874,2026-05-31T17:38:41.322971+00:00


Completed requests: 2508 / 2508


## 7. Download Completed Batch Outputs

This is guarded so Run All does not download files unexpectedly. Set `MANUAL_DOWNLOAD = True` after the batch status is `completed`.

In [11]:
MANUAL_DOWNLOAD = True
RUN_ID_TO_DOWNLOAD = None  # None means latest run_id from manifest.

if MANUAL_DOWNLOAD:
    downloaded_result = rec.download_completed_embedding_batch_outputs(run_id=RUN_ID_TO_DOWNLOAD)
    print(json.dumps(downloaded_result, ensure_ascii=False, indent=2))
else:
    print("MANUAL_DOWNLOAD is False. Set it to True after the batch is completed.")

Already downloaded: batch 1 -> location_embedding_output_20260531T173837Z_part_001_0_2508.jsonl
{
  "run_id": "20260531T173837Z",
  "downloaded": [
    {
      "run_id": "20260531T173837Z",
      "batch_number": 1,
      "batch_id": "batch_6a1c7220819c81908706c380752a4ce9",
      "input_file_id": "file-32fhn4W6VBPUCbqafwDifG",
      "jsonl_path": "/Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/embedding_requests/location_embedding_requests_20260531T173837Z_part_001_0_2508.jsonl",
      "row_start": 0,
      "row_end": 2508,
      "n_inputs": 2508,
      "estimated_input_tokens": 1287392,
      "estimated_batch_cost_usd": 0.01287392,
      "model": "text-embedding-3-small",
      "dimensions": null,
      "endpoint": "/v1/embeddings",
      "source_csv_path": "/Users/ilya/Documents/VisualStudioCode/SLOCO/data_scraping/output/ai_location_summaries/final/final_ai_dataframe_with_map_scores_latest.csv",
      "submitted_at_utc": "2026-05-31T1

## 8. Assemble Or Load Embedding Store

After downloading outputs, assemble a dense `.npy` embedding matrix and metadata. In later sessions, load the latest store directly.

In [12]:
MANUAL_ASSEMBLE = True
RUN_ID_TO_ASSEMBLE = None  # None means latest run_id from manifest.

if MANUAL_ASSEMBLE:
    df_with_embeddings, location_embedding_matrix, location_embedding_metadata, matrix_path, metadata_path = rec.assemble_location_embedding_store(
        df_prepared,
        run_id=RUN_ID_TO_ASSEMBLE,
    )
    print("Matrix:", matrix_path)
    print("Metadata:", metadata_path)
else:
    print("MANUAL_ASSEMBLE is False. Set it to True after downloading completed batch outputs.")

Could not save parquet metadata, saved CSV instead: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.
embedding_matrix shape: (2508, 1536)
saved matrix: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/embedding_store/location_embeddings_20260531T173837Z.npy
saved metadata: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/embedding_store/location_embeddings_20260531T173837Z_metadata.csv
rows with embeddings: 2508 / 2508
Matrix: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_loca

In [13]:
# Load latest existing embedding store. Run after you have assembled at least one store.
try:
    location_embedding_matrix, location_embedding_metadata = rec.load_location_embedding_store()
    df_with_embeddings = rec.attach_embedding_refs(df_prepared, location_embedding_metadata)
    print("Loaded embedding matrix:", location_embedding_matrix.shape)
    print("Rows with embeddings:", int(df_with_embeddings["has_embedding"].sum()), "/", len(df_with_embeddings))
    display(location_embedding_metadata.head())
except Exception as exc:
    print(type(exc).__name__, str(exc))
    print("No embedding store loaded yet. Submit, download, and assemble a batch first.")

Loaded embedding matrix: (2508, 1536)
Rows with embeddings: 2508 / 2508


,source_row_index,place_id,custom_id,embedding_row,has_embedding,embedding_text_hash,error,usage_total_tokens
0,0,ChIJaX6OBQD_sUARpsOX3IMt2R4,location_0_ChIJaX6OBQD_sUARpsOX3IMt2R4,0,True,e86e286ebaca328affe102cf956b4b9e1cf08c6a8e74bef0bcdeb3dac3b2bf02,NaN,416
1,1,ChIJ_xaqNQMCskAR8aQ8oHos1Ro,location_1_ChIJ_xaqNQMCskAR8aQ8oHos1Ro,1,True,850254ad55dc1e94db644e82524bed12684fe8389e36c9f626956ad20b8fe8b4,NaN,568
2,2,ChIJnUA9FgD_sUAR2ooJ3CWb0Zc,location_2_ChIJnUA9FgD_sUAR2ooJ3CWb0Zc,2,True,1a03c43358d8e8b754b69d7f5501c15f69b551625ec8a8ff120c33eea4a9900e,NaN,476
3,3,ChIJAQAckz__sUARPQnYMKtxIRQ,location_3_ChIJAQAckz__sUARPQnYMKtxIRQ,3,True,059da87008f78ebbd452de6b4bf2bf4eb6ee96fa83869f49275cdd498cac9087,NaN,628
4,4,ChIJbYXdJwD_sUAR2yjbAGOKJPM,location_4_ChIJbYXdJwD_sUAR2yjbAGOKJPM,4,True,dc81af8a188a974b05c637645ae8577546eba6121daa8f036661b548f000ae18,NaN,560


## 9. Build Candidate Dataset

In [14]:
if "df_with_embeddings" not in globals() or "location_embedding_matrix" not in globals():
    raise RuntimeError("Load or assemble embeddings before building the recommender dataset.")

df_model_all = rec.get_rows_with_embeddings(df_with_embeddings).copy()
df_model_all["quality_score"] = rec.compute_quality_score(df_model_all)

df_candidates_default = rec.build_candidate_dataset(
    df_with_embeddings,
    min_map_visibility_score=20,
    exclude_low_confidence=True,
)
df_candidates_default, candidate_embedding_matrix = rec.compact_embedding_matrix_for_rows(
    df_candidates_default,
    location_embedding_matrix,
)

print("Rows with embeddings:", len(df_model_all))
print("Default candidate rows:", len(df_candidates_default))
print("Candidate embedding matrix:", candidate_embedding_matrix.shape)
display(df_candidates_default[["name", "ai_place_type_summary", "ai_tags_csv", "google_rating", "map_visibility_score", "quality_score"]].head(10))

Rows with embeddings: 2508
Default candidate rows: 2022
Candidate embedding matrix: (2022, 1536)


,name,ai_place_type_summary,ai_tags_csv,google_rating,map_visibility_score,quality_score
1,"""Seneca Anticafe""",bookstore coworking cafe,"cozy,quiet,laptop_friendly,study_friendly,work_friendly,solo_friendly,group_friendly,local_favorite,specialty_coffee,tea,good_value,hard_to_park",4.8,89.9,0.895822
3,#ALTFEL,lounge bar,"lively,loud,crowded,casual,local_favorite,group_friendly,good_for_night_out,good_for_daytime,outdoor_seating,reservation_recommended,drinks_focused,cheap,good_value,cocktails,live_music,friendly_staff,inconsistent_service,slow_service",4.2,65.1,0.712999
4,#Altfel Park Lake,terrace bar,"casual,cozy,lively,terrace,cocktails,drinks_focused,good_value,friendly_staff,inconsistent_service,good_for_daytime,good_for_night_out",3.7,33.1,0.439590
5,#aici,riverfront lounge bar,"outdoor_seating,terrace,nice_view,lively,romantic,group_friendly,good_for_daytime,good_for_night_out,cocktails,food_focused,friendly_staff,slow_service,inconsistent_service,premium,expensive,smoking_area,crowded",4.1,64.1,0.716572
6,#altfel Unirii,lounge bar,"lively,loud,casual,local_favorite,group_friendly,outdoor_seating,cocktails,drinks_focused,good_value,good_for_night_out,inconsistent_service,cheap",4.0,55.8,0.642436
8,.,park terrace bar,"outdoor_seating,quiet,cozy,good_for_daytime,drinks_focused,good_value,good_coffee,wine,craft_beer,local_favorite,laptop_friendly,hard_to_park",4.4,60.2,0.668697
9,1 Minute Floreasca Park,convenience store cafe,"good_value,friendly_staff,food_focused,good_for_daytime,breakfast,good_coffee,casual,fast_service,cheap",4.4,55.4,0.613615
10,1 Minute Global City,coffee shop,"good_coffee,friendly_staff,fast_service,good_value,casual,good_for_daytime,wheelchair_accessible",4.1,51.2,0.571171
11,1 minute,coffee shop,"good_coffee,casual,local_favorite,good_for_daytime,good_value,friendly_staff,inconsistent_service",4.0,47.3,0.531040
12,1 minute,cafe,"good_coffee,brunch,good_value,casual,lively,food_focused,good_for_daytime,fast_service",3.9,33.6,0.448144


## 10. Global Clustering

In [16]:
# Evaluate several k values on the default candidate pool.
kmeans_eval_df = rec.evaluate_kmeans_clusters(
    candidate_embedding_matrix,
    k_values=range(3, 15),
    sample_size=min(2500, len(candidate_embedding_matrix)),
)
display(kmeans_eval_df)

GLOBAL_N_CLUSTERS = int(kmeans_eval_df.iloc[0]["k"]) if not kmeans_eval_df.empty else 8
print("Selected GLOBAL_N_CLUSTERS:", GLOBAL_N_CLUSTERS)

,k,silhouette_cosine
0,8,0.111991
1,6,0.110759
2,14,0.106421
3,7,0.104961
4,13,0.103174
5,11,0.101467
6,9,0.100958
7,5,0.097740
8,3,0.090388
9,4,0.089233


Selected GLOBAL_N_CLUSTERS: 8


In [17]:
global_cluster_model, global_cluster_labels = rec.fit_global_location_clusters(
    candidate_embedding_matrix,
    n_clusters=GLOBAL_N_CLUSTERS,
)

df_candidates_default = df_candidates_default.copy()
df_candidates_default["global_cluster"] = global_cluster_labels.astype(int)

cluster_summary_df = rec.summarize_global_clusters(df_candidates_default)
display(cluster_summary_df)

,global_cluster,rows,top_primary_types,top_tags,examples,median_quiet_lively,median_work_social,median_day_night,median_casual_premium,median_drinks_food,median_local_tourist,median_cheap_expensive,median_traditional_experimental
0,0,250,"coffee_shop (141), cafe (109)","good_value (235), good_coffee (228), good_for_daytime (207), friendly_staff (194), cheap (187), casual (187), local_favorite (177), specialty_coffee (175), fast_service (146), solo_friendly (115)",5 To Go; 5 To Go PLUS; 5 to go; 5 to go - Favorit; 5 To Go,30.0,25.0,10.0,15.0,20.0,15.0,10.0,18.0
1,1,205,"bar (54), restaurant (18), lounge_bar (12), casino (11), pub (9), gastropub (9)","local_favorite (123), good_for_night_out (119), group_friendly (112), lively (112), friendly_staff (98), casual (96), good_value (93), cocktails (91), good_for_daytime (88), outdoor_seating (84)",Jeonjuu Korean BBQ; El Dictador; Ganesha Caffe Primaverii; Hop Hooligans Taproom; Tap Craft Beer,63.0,72.0,65.0,40.0,28.0,20.0,52.0,22.0
2,2,505,"bar (136), restaurant (124), pub (35), gastropub (29), cafe (20), wine_bar (18)","local_favorite (424), outdoor_seating (382), good_value (354), good_for_night_out (291), group_friendly (290), good_for_daytime (269), food_focused (269), cocktails (262), casual (259), cozy (256)",Crama La Sorinescu; SIPPERS Bucharest Cocktail Bar; Restaurant Slow; KAO Chinese Restaurant and Private Karaoke; Nishani | A fi sau a nu fi,58.0,72.0,56.0,38.0,55.0,22.0,46.0,22.0
3,3,32,"coffee_shop (14), cafe (13), fast_food_restaurant (5)","good_for_daytime (31), outdoor_seating (26), good_coffee (24), friendly_staff (21), cozy (21), desserts (21), breakfast (20), casual (20), local_favorite (20), specialty_coffee (18)",Starbucks; Starbucks insula; Starbucks; Starbucks; Starbucks,43.5,43.5,20.0,34.0,29.0,23.5,52.0,18.0
4,4,216,"coffee_shop (89), cafe (41), dessert_shop (17), ice_cream_shop (10), supermarket (5), amusement_center (5)","good_for_daytime (176), good_coffee (151), good_value (146), friendly_staff (138), local_favorite (130), specialty_coffee (124), casual (112), cozy (101), desserts (87), quiet (82)","Friddi; ""Seneca Anticafe""; Bebe Cafe; The Cone; Velocità",28.0,30.0,15.0,28.0,45.0,18.0,35.0,20.0
5,5,23,"gas_station (22), auto_parts_store (1)","friendly_staff (22), good_for_daytime (20), fast_service (15), good_coffee (15), wheelchair_accessible (15), inconsistent_service (13), good_value (12), casual (11), local_favorite (8), good_for_night_out (7)",Rompetrol; Rompetrol; Rompetrol; Rompetrol; Rompetrol,42.0,20.0,30.0,20.0,20.0,15.0,62.0,8.0
6,6,412,"coffee_shop (326), cafe (79), brunch_restaurant (2), bar (1), bistro (1), restaurant (1)","specialty_coffee (388), good_coffee (367), local_favorite (354), cozy (344), good_value (340), friendly_staff (327), quiet (294), good_for_daytime (291), solo_friendly (220), outdoor_seating (198)",God's Coffee -Cafea de Specialitate; Black Coffee Amzei; Fauna Coffee Shop; Two Minutes; Custom CoffeeShop,25.0,30.0,12.0,35.0,20.0,18.0,25.0,24.5
7,7,379,"coffee_shop (246), cafe (74), restaurant (6), bakery (5), brunch_restaurant (5), bistro (4)","local_favorite (316), good_coffee (311), specialty_coffee (307), good_value (287), cozy (287), friendly_staff (271), good_for_daytime (268), quiet (201), outdoor_seating (185), desserts (175)",COFFETEA Piața Romană; Cărturești Carusel; Camera din Față; Saint Roastery; boteca13,28.0,35.0,15.0,35.0,25.0,20.0,35.0,22.0


In [18]:
cluster_viz_df = rec.create_embedding_projection(
    df_candidates_default,
    candidate_embedding_matrix,
    max_points=None,
)

fig = rec.plot_global_clusters(cluster_viz_df)
fig.show()

## 11. Helpers For Choosing Favorites

In [19]:
# Example lookup. Change the query to find real favorites by name.
rec.find_locations_by_name("coffee", df_candidates_default, limit=20)

,place_id,name,primary_type,ai_place_type_summary,ai_card_summary,ai_tags_csv,google_rating,map_visibility_score,global_cluster
15,ChIJF_WM2fQDskAR5OqyAz_9Loo,13 - Cafenea cu Poezie - Specialty Coffee,cafe,specialty coffee shop,"A small, poetry-filled specialty coffee shop with excellent coffee, a cozy calm atmosphere, and a welcoming terrace for slow mornings.","cozy,quiet,specialty_coffee,good_coffee,local_favorite,solo_friendly,date_spot,outdoor_seating,laptop_friendly,desserts,vegetarian_options,friendly_staff",4.9,78.8,7
18,ChIJ1Tq3cDsDskARjOzp9wO9pT4,146 I Love Coffee,coffee_shop,coffee shop,"Friendly neighborhood coffee shop for coffee and dessert, with a calm dine-in feel and easy takeout on weekdays.","specialty_coffee,good_coffee,desserts,casual,quiet,good_for_daytime,solo_friendly,dog_friendly,good_value",4.7,51.2,4
24,ChIJb6CQZBr_sUARtU0kT_UH41o,20grams Coffee Roasters,coffee_roastery,specialty coffee roastery and cafe,"Specialty coffee roastery with excellent espresso drinks, fresh sweets, and a calm, design-forward room that works well for a relaxed stop.","specialty_coffee,good_coffee,cozy,quiet,laptop_friendly,local_favorite,good_for_daytime,food_focused,desserts,good_value,friendly_staff,inconsistent_service,dog_friendly,wheelchair_accessible,hard_to_park",4.8,85.7,4
26,ChIJ52NZYwADskARZRgmcwjkuC8,232 Specialty COFFEE SHOP,coffee_shop,specialty coffee shop,"Small, cozy specialty coffee shop with excellent espresso, a friendly owner, and a sunny terrace that feels calm and welcoming.","specialty_coffee,good_coffee,cozy,quiet,friendly_staff,local_favorite,small_space,outdoor_seating,dog_friendly,good_for_daytime,breakfast,desserts,cheap,good_value",5.0,76.1,6
27,ChIJH31Hi4v5sUARcgO_c6lAphY,249Pantelimon Coffee Shop (Damiani Coffee),coffee_shop,specialty coffee shop,"A warm neighborhood coffee shop with specialty drinks, friendly service, and a calm, welcoming feel that makes it easy to linger.","specialty_coffee,good_coffee,cozy,quiet,small_space,friendly_staff,good_value,outdoor_seating,dog_friendly,family_friendly,good_for_daytime,no_wait",5.0,75.1,7
30,ChIJDb7FSSn5sUARRhwXqqgy6i8,4COFFEE,coffee_shop,specialty coffee shop,"A cozy specialty coffee shop with a calm courtyard feel, excellent matcha and coffee, and a welcoming vibe for quiet breaks.","quiet,cozy,stylish,local_favorite,specialty_coffee,good_coffee,tea,desserts,outdoor_seating,good_for_daytime,friendly_staff,laptop_friendly",4.9,77.1,6
32,ChIJ5cQSv8X_sUART3znhMJSbxg,4You Coffee Shop,cafe,neighborhood coffee shop,"A cozy neighborhood coffee shop with strong cappuccinos, friendly service, and very affordable prices, best for a quiet daytime break.","cozy,quiet,local_favorite,specialty_coffee,good_coffee,cheap,good_value,friendly_staff,solo_friendly,laptop_friendly,outdoor_seating,good_for_daytime",4.8,67.9,6
328,ChIJAcI2FVoDskARm1bm_gLPYb8,ADESSO Specialty Coffee,cafe,specialty coffee cafe,"A cozy specialty coffee spot with good drinks, brunch bites, and a pleasant terrace, best for relaxed daytime coffee stops.","cozy,specialty_coffee,good_coffee,brunch,good_for_daytime,work_friendly,laptop_friendly,outdoor_seating,dog_friendly,good_value,friendly_staff,inconsistent_service",4.5,70.2,6
331,ChIJxzlz8J3_sUARVOtnOAD4CMM,ALT COFFEE,coffee_shop,specialty coffee shop,"A small, cheerful coffee shop with great espresso, a chill vibe, and a social-mission feel, best for a relaxed daytime stop.","specialty_coffee,good_coffee,cozy,quiet,casual,local_favorite,good_for_daytime,small_space,friendly_staff,specialty_coffee",4.7,61.7,6
333,ChIJpUMcapQDskARjwIIJvpVUxw,AM PM Specialty Coffee Bucuresti,cafe,specialty coffee shop,"Cozy specialty coffee shop with creative drinks, a quiet intimate feel, and a leafy terrace that works well for a relaxed break or date.","cozy,quiet,specialty_coffee,good_coffee,desserts,cocktails,outdoor_seating,garden,date_spot,laptop_friendly,good_for_daytime,small_space,friendly_staff,good_value,expensive,inconsistent_service,hard_to_park"

## 12. Simulate User Favorites And Recommend

Replace `example_favorite_place_ids` with real user-selected `place_id` values when available. If it is empty, the notebook samples 12 favorites randomly from the candidate pool.

In [20]:
example_favorite_place_ids = []

if example_favorite_place_ids:
    user_favorites = rec.get_favorite_rows(df_candidates_default, favorite_place_ids=example_favorite_place_ids)
else:
    example_favorite_indices = df_candidates_default.sample(n=12, random_state=7).index.tolist()
    user_favorites = rec.get_favorite_rows(df_candidates_default, favorite_indices=example_favorite_indices)

user_favorites = rec.cluster_user_favorites(
    user_favorites,
    candidate_embedding_matrix,
    max_profile_clusters=4,
)

print("Selected favorite locations:")
display(user_favorites[[
    "profile_cluster", "name", "ai_place_type_summary", "ai_card_summary",
    "ai_tags_csv", "google_rating", "map_visibility_score", "global_cluster", "place_id",
]].sort_values(["profile_cluster", "name"]))

Selected favorite locations:


,profile_cluster,name,ai_place_type_summary,ai_card_summary,ai_tags_csv,google_rating,map_visibility_score,global_cluster,place_id
122,0,5 To Go,coffee shop,"A friendly, budget-friendly coffee stop with strong espresso, fast service, and a few seats for a quick break or takeaway.","specialty_coffee,good_coffee,cheap,good_value,friendly_staff,fast_service,casual,solo_friendly,laptop_friendly,work_friendly,good_for_daytime,crowded",4.8,73.3,0,ChIJqagCgrwBskARyUgvvOR_a60
156,0,5 to GO Victor Brauner,budget coffee shop,"Budget coffee stop with a bright, casual feel and friendly morning energy, best for a quick low-cost cup rather than a lingered sit-down.","cheap,good_value,casual,local_favorite,good_for_daytime,specialty_coffee,friendly_staff,inconsistent_service,slow_service,wheelchair_accessible,hard_to_park",4.0,51.0,0,ChIJY43OTwD9sUARUNcWl0VwwOo
192,0,5 to go,coffee shop,"A small, inexpensive coffee stop with friendly service and a bright, casual feel, best for a quick latte or takeaway drink.","good_coffee,specialty_coffee,cheap,good_value,friendly_staff,inconsistent_service,food_focused,laptop_friendly,casual,local_favorite,good_for_daytime,small_space",4.4,62.4,0,ChIJUZfrOQP5sUARFn-p-t_0eFg
780,0,Coffee Store,coffee shop,"A lively central coffee shop with good cappuccino, banana bread, and a friendly terrace, better for coffee and drinks than a full meal.","specialty_coffee,good_coffee,cozy,lively,casual,local_favorite,laptop_friendly,study_friendly,solo_friendly,group_friendly,outdoor_seating,good_for_daytime,drinks_focused,food_focused,good_value,inconsistent_service",4.4,71.2,6,ChIJx87E5cT_sUAR8vnu9sO-40g
864,0,D'Ice Sweets,tea and dessert cafe,"Fresh-made teas and fruit drinks in a small, friendly café with parking and a bright daytime feel.","tea,desserts,good_value,friendly_staff,good_for_daytime,family_friendly,easy_parking,vegetarian_options",5.0,50.2,7,ChIJvbUJuej3sUARgTeznVl96io
954,0,Elinor coffee break,coffee shop,"Friendly coffee stop with tasty snacks, hot drinks, and a quiet, easygoing feel that works well for a quick break or takeout.","friendly_staff,quiet,good_value,cheap,good_coffee,food_focused,casual,local_favorite,no_wait,good_for_daytime,solo_friendly",5.0,53.4,6,ChIJiV2YP7UDskARjhQv_zLH5YE
1661,0,Oficiul 1,speakeasy cocktail bar,"Hidden speakeasy-style cocktail bar with inventive drinks, a warm buzz, and a post-office setting that feels intimate and memorable.","cocktails,lively,romantic,hidden_gem,date_spot,group_friendly,reservation_recommended,specialty_coffee,friendly_staff,inconsistent_service,small_space,good_for_night_out",4.6,75.8,2,ChIJubXHIUf_sUARZ1GYK2a340g
1764,0,Pit’s coffee shop,coffee shop,"A cheerful coffee stop for strong lattes, cookies, and nuts, with fast takeout, friendly service, and very good value.","specialty_coffee,good_coffee,desserts,cheap,good_value,friendly_staff,fast_service,casual,local_favorite,good_for_daytime,work_friendly,solo_friendly",5.0,63.2,6,ChIJgQfwmzL_sUARfC4bwCvgXXY
1951,0,Saint Roastery Baneasa,specialty coffee shop,"Specialty coffee spot in Băneasa with a cozy, modern feel, strong espresso drinks, and a calm stop for shopping breaks or studying.","specialty_coffee,cozy,quiet,laptop_friendly,study_friendly,solo_friendly,group_friendly,good_coffee,desserts,good_value,friendly_staff,local_favorite",4.8,70.6,7,ChIJ8dfN3CYDskARxseZ0fsu9ME
2002,0,Sister's caffe and bistro,coffee shop and dessert cafe,"Chic, quiet cafe with impeccable service, fresh cakes, and strong coffee—an easy pick for a relaxed daytime treat.","stylish,quiet,good_value,specialty_coffee,desserts,good_for_daytime,friendly_staff,no_wait",5.0,52.5,6,ChIJS8ZIx2f_rUARzt59Xp4Omw8


In [21]:
pure_recommendations = rec.recommend_for_profile_clusters(
    df_candidates_default,
    candidate_embedding_matrix,
    user_favorites,
    total_n=100,
    mode="pure",
)

hybrid_recommendations = rec.recommend_for_profile_clusters(
    df_candidates_default,
    candidate_embedding_matrix,
    user_favorites,
    total_n=100,
    mode="hybrid",
)

print("Pure similarity recommendations:", len(pure_recommendations))
display(rec.display_recommendation_columns(pure_recommendations).head(30))

print("Hybrid recommendations:", len(hybrid_recommendations))
display(rec.display_recommendation_columns(hybrid_recommendations).head(30))

Pure similarity recommendations: 100


,recommendation_rank,profile_cluster,recommendation_score,similarity,tag_overlap,axis_similarity,quality_score,price_match,name,ai_place_type_summary,ai_card_summary,ai_tags_csv,google_rating,google_user_rating_count,map_visibility_score,based_on_favorites,place_id
0,1,0,0.957513,0.957513,0.270270,0.915114,0.676984,0.922727,5 to go,coffee shop,"A small, friendly coffee stop with excellent coffee, fast service, and an easygoing atmosphere that works well for quick breaks or relaxed catch-ups.","specialty_coffee,good_coffee,casual,local_favorite,friendly_staff,fast_service,quiet,good_for_daytime,outdoor_seating,spacious,good_value,inconsistent_service",4.7,26.0,60.3,Pit’s coffee shop; 5 to GO Victor Brauner; 5 To Go; Coffee Store; D'Ice Sweets; Utopia Coffee Bar; Elinor coffee break; Sister's caffe and bistro,ChIJwxYRVAD9sUAR8Xaz3FaUuoo
1,2,0,0.957333,0.957333,0.270270,0.884886,0.715561,0.862727,5 to go,coffee shop,"A friendly, low-key coffee stop with good espresso, fast service, and a clean, easygoing atmosphere.","specialty_coffee,good_coffee,casual,local_favorite,quiet,fast_service,friendly_staff,good_value,good_for_daytime,small_space,wheelchair_accessible,easy_parking",4.7,52.0,65.7,Pit’s coffee shop; 5 to GO Victor Brauner; 5 To Go; Coffee Store; D'Ice Sweets; Utopia Coffee Bar; Elinor coffee break; Sister's caffe and bistro,ChIJ7S62y_8BskARKniGs71mRYI
2,3,0,0.951351,0.951351,0.305556,0.913636,0.608938,0.842727,5 to go,coffee shop,"A small, friendly coffee stop with good coffee, quick service, and a calm feel, though opening hours can be unreliable.","specialty_coffee,good_coffee,casual,quiet,local_favorite,friendly_staff,fast_service,good_value,cheap,good_for_daytime,work_friendly,small_space",4.3,28.0,55.1,Pit’s coffee shop; 5 to GO Victor Brauner; 5 To Go; Coffee Store; D'Ice Sweets; Utopia Coffee Bar; Elinor coffee break; Sister's caffe and bistro,ChIJUahUIQD_sUARJNtnAI7C7qA
3,4,0,0.949681,0.949681,0.305556,0.881477,0.461269,0.842727,5 to go,budget coffee shop,"A small, inexpensive coffee stop with decent coffee and a cozy feel, but service can be inconsistent and sometimes sharply unwelcoming.","cheap,good_value,specialty_coffee,good_coffee,desserts,casual,cozy,local_favorite,solo_friendly,work_friendly,good_for_daytime,friendly_staff,inconsistent_service,no_wait,easy_parking",3.7,36.0,32.2,Pit’s coffee shop; 5 to GO Victor Brauner; 5 To Go; Coffee Store; D'Ice Sweets; Utopia Coffee Bar; Elinor coffee break; Sister's caffe and bistro,ChIJP2pAHp8DrkARc6ytDhNCRiE
4,5,0,0.949086,0.949086,0.222222,0.898409,0.447648,0.842727,5 to go,chain coffee shop,"Budget-friendly coffee stop with outdoor seating and quick takeaway, but coffee quality and service can be inconsistent.","cheap,good_value,outdoor_seating,casual,local_favorite,laptop_friendly,solo_friendly,group_friendly,good_for_daytime,good_coffee,specialty_coffee,inconsistent_service",3.6,60.0,30.0,Pit’s coffee shop; 5 to GO Victor Brauner; 5 To Go; Coffee Store; D'Ice Sweets; Utopia Coffee Bar; Elinor coffee break; Sister's caffe and bistro,ChIJEQWPoScDskARBgS9NB4fVHQ
5,6,0,0.948876,0.948876,0.277778,0.884659,0.751003,0.822727,5 to go,budget coffee shop,"Cheap, quick coffee spot with a cozy terrace and friendly moments, but service and opening hours can be unreliable.","cheap,good_value,cozy,local_favorite,outdoor_seating,good_for_daytime,specialty_coffee,desserts,laptop_friendly,friendly_staff,inconsistent_service,crowded",4.5,267.0,70.8,Pit’s coffee shop; 5 to GO Victor Brauner; 5 To Go; Coffee Store; D'Ice Sweets; Utopia Coffee Bar; Elinor coffee break; Sister's caffe and bistro,ChIJw0RN-v7_sUAR5dDqbN48wEg
6,7,0,0.948657,0.948657,0.305556,0.926136,0.776697,0.942727,Some Coffee,coffee shop,"A cozy neighborhood coffee shop with friendly baristas, quiet seating, and well-liked coffee, matcha, and low-key hangout space.","cozy,quiet,local_favorite,solo_friendly,date_spot,specialty_coffee,good_coffee,tea,desserts,good_value,friendly_staff,work_friendly",5.0,59.0,73.2,Pit

Hybrid recommendations: 98


,recommendation_rank,profile_cluster,recommendation_score,similarity,tag_overlap,axis_similarity,quality_score,price_match,name,ai_place_type_summary,ai_card_summary,ai_tags_csv,google_rating,google_user_rating_count,map_visibility_score,based_on_favorites,place_id
0,1,0,0.893713,0.922666,0.342105,0.956136,0.887625,0.957273,Custom CoffeeShop,specialty coffee shop,"Specialty coffee shop with a cozy garage vibe, strong espresso and cold brew, friendly service, and a relaxed terrace for coffee or a light bite.","specialty_coffee,good_coffee,cozy,quiet,local_favorite,hidden_gem,good_for_daytime,breakfast,brunch,desserts,cocktails,outdoor_seating,dog_friendly,friendly_staff,good_value",5.0,326.0,89.1,Pit’s coffee shop; 5 to GO Victor Brauner; 5 To Go; Coffee Store; D'Ice Sweets; Utopia Coffee Bar; Elinor coffee break; Sister's caffe and bistro,ChIJxyO5RAADskAROM_fWiI1Xvs
1,2,0,0.891681,0.941010,0.361111,0.913864,0.785419,0.957273,I’m coffee,cozy neighborhood coffee shop,"A small, cozy coffee shop with friendly service, good espresso, and a quiet courtyard feel that works well for a relaxed break or morning stop.","cozy,quiet,local_favorite,solo_friendly,work_friendly,laptop_friendly,dog_friendly,outdoor_seating,specialty_coffee,good_coffee,desserts,good_value,friendly_staff",4.8,124.0,75.2,Pit’s coffee shop; 5 to GO Victor Brauner; 5 To Go; Coffee Store; D'Ice Sweets; Utopia Coffee Bar; Elinor coffee break; Sister's caffe and bistro,ChIJL-erFgIDskARi0uWPBKXqKw
2,3,0,0.891468,0.915693,0.526316,0.915341,0.726303,0.837273,Green Caffe,coffee shop and cafe,"A cozy, stylish cafe for good coffee, cakes, and relaxed daytime hangs, with outdoor seating and a pleasant evening wine or cocktail option.","cozy,stylish,local_favorite,laptop_friendly,study_friendly,solo_friendly,group_friendly,outdoor_seating,specialty_coffee,good_coffee,tea,desserts,breakfast,cocktails,wine,friendly_staff,fast_service,quiet,good_for_daytime,good_for_night_out,good_value,inconsistent_service",4.5,146.0,67.6,Pit’s coffee shop; 5 to GO Victor Brauner; 5 To Go; Coffee Store; D'Ice Sweets; Utopia Coffee Bar; Elinor coffee break; Sister's caffe and bistro,ChIJAVpHFQD_sUAR17joK2qKCPs
3,4,0,0.891190,0.915201,0.432432,0.946136,0.810937,0.867273,Cafeteca,coffee shop,"A cozy, plant-filled coffee shop with good coffee, pastries, and a calm hangout feel, best for reading, chatting, or working quietly.","cozy,quiet,local_favorite,laptop_friendly,study_friendly,solo_friendly,date_spot,group_friendly,outdoor_seating,specialty_coffee,good_coffee,desserts,brunch,tea,good_for_daytime,friendly_staff,good_value",4.7,330.0,78.8,Pit’s coffee shop; 5 to GO Victor Brauner; 5 To Go; Coffee Store; D'Ice Sweets; Utopia Coffee Bar; Elinor coffee break; Sister's caffe and bistro,ChIJtXqptLb_sUAR-AA_-860on4
4,5,0,0.890404,0.932060,0.351351,0.946136,0.796499,0.942727,232 Specialty COFFEE SHOP,specialty coffee shop,"Small, cozy specialty coffee shop with excellent espresso, a friendly owner, and a sunny terrace that feels calm and welcoming.","specialty_coffee,good_coffee,cozy,quiet,friendly_staff,local_favorite,small_space,outdoor_seating,dog_friendly,good_for_daytime,breakfast,desserts,cheap,good_value",5.0,77.0,76.1,Pit’s coffee shop; 5 to GO Victor Brauner; 5 To Go; Coffee Store; D'Ice Sweets; Utopia Coffee Bar; Elinor coffee break; Sister's caffe and bistro,ChIJ52NZYwADskARZRgmcwjkuC8
5,6,0,0.889458,0.936334,0.378378,0.919886,0.773810,0.892727,Coffee Shop,neighborhood coffee shop,"A cozy neighborhood coffee shop with strong espresso, fair prices, and a relaxed terrace that works well for studying or an easy break.","cozy,local_favorite,laptop_friendly,study_friendly,solo_friendly,family_friendly,dog_friendly,outdoor_seating,specialty_coffee,good_coffee,breakfast,desserts,good_value,friendly_staff,quiet",4.7,151.0,73.8,Pit’s coffee shop; 5 to GO Victor Brauner; 5 To Go; Coffee Store; D'Ice Sweets; Utopia Coffee Bar; Elinor coffee break; Sister's caffe and bistro,ChIJAePl2eT9sUARdWC1sAjPz3g


In [22]:
comparison_cols = ["place_id", "name", "profile_cluster", "recommendation_rank", "recommendation_score", "similarity"]
comparison = (
    rec.display_recommendation_columns(pure_recommendations)[comparison_cols]
    .rename(columns={"recommendation_rank": "pure_rank", "recommendation_score": "pure_score", "similarity": "pure_similarity"})
    .merge(
        rec.display_recommendation_columns(hybrid_recommendations)[comparison_cols]
        .rename(columns={"recommendation_rank": "hybrid_rank", "recommendation_score": "hybrid_score", "similarity": "hybrid_similarity"}),
        on=["place_id", "name", "profile_cluster"],
        how="outer",
    )
    .sort_values(["profile_cluster", "hybrid_rank", "pure_rank"], na_position="last")
)

display(comparison.head(50))

,place_id,name,profile_cluster,pure_rank,pure_score,pure_similarity,hybrid_rank,hybrid_score,hybrid_similarity
160,ChIJxyO5RAADskAROM_fWiI1Xvs,Custom CoffeeShop,0,NaN,NaN,NaN,1.0,0.893713,0.922666
59,ChIJL-erFgIDskARi0uWPBKXqKw,I’m coffee,0,32.0,0.941010,0.941010,2.0,0.891681,0.941010
26,ChIJAVpHFQD_sUAR17joK2qKCPs,Green Caffe,0,NaN,NaN,NaN,3.0,0.891468,0.915693
148,ChIJtXqptLb_sUAR-AA_-860on4,Cafeteca,0,NaN,NaN,NaN,4.0,0.891190,0.915201
12,ChIJ52NZYwADskARZRgmcwjkuC8,232 Specialty COFFEE SHOP,0,NaN,NaN,NaN,5.0,0.890404,0.932060
27,ChIJAePl2eT9sUARdWC1sAjPz3g,Coffee Shop,0,46.0,0.936334,0.936334,6.0,0.889458,0.936334
70,ChIJNxLWARX_sUARhPXefSehv1g,Some Coffee,0,7.0,0.948657,0.948657,7.0,0.888814,0.948657
137,ChIJoXyi4af_sUARnetPRisyC9A,Vicii Coffee Shop,0,NaN,NaN,NaN,8.0,0.887379,0.923262
62,ChIJLcGb3V75sUAR-XwnSwwd5_s,5 To Go,0,20.0,0.943752,0.943752,9.0,0.885710,0.943752
149,ChIJtYTIXzP_sUAR4E6tTHEkpc8,The Coffee Shop,0,NaN,NaN,NaN,10.0,0.885523,0.935116


## 13. Visualize Favorites And Recommendations

In [23]:
profile_fig = rec.plot_profile_recommendations(
    df_candidates_default,
    candidate_embedding_matrix,
    user_favorites,
    hybrid_recommendations,
    background_sample=2000,
)
profile_fig.show()

## 14. Export Model Tables And Recommendations

In [24]:
export_paths = rec.save_recommender_exports(
    df_model_all=df_model_all,
    df_candidates_default=df_candidates_default,
    pure_recommendations=pure_recommendations,
    hybrid_recommendations=hybrid_recommendations,
)

for name, path in export_paths.items():
    print(f"{name}: {path}")

model_all: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/exports/df_model_all_20260531T182821Z.csv
candidates_default: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/exports/df_candidates_default_20260531T182821Z.csv
pure_recommendations: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/exports/pure_recommendations_20260531T182821Z.csv
hybrid_recommendations: /Users/ilya/Documents/VisualStudioCode/SLOCO/recommendation_system/ai_location_recommender/data/exports/hybrid_recommendations_20260531T182821Z.csv


## 15. Smoke Test Helper

Use this after embeddings are loaded to validate that the recommender returns enough rows and never recommends selected favorites back to the user.

In [25]:
smoke_favorites, smoke_pure_recs, smoke_hybrid_recs = rec.run_recommender_smoke_test(
    df_candidates_default,
    candidate_embedding_matrix,
    n_favorites=12,
    total_n=50,
    random_state=11,
)

print("Smoke favorites:", len(smoke_favorites))
print("Smoke pure recommendations:", len(smoke_pure_recs))
print("Smoke hybrid recommendations:", len(smoke_hybrid_recs))
display(smoke_favorites[["profile_cluster", "name", "ai_place_type_summary", "ai_tags_csv"]].sort_values("profile_cluster"))
display(rec.display_recommendation_columns(smoke_hybrid_recs).head(20))

Smoke favorites: 12
Smoke pure recommendations: 50
Smoke hybrid recommendations: 50


,profile_cluster,name,ai_place_type_summary,ai_tags_csv
918,0,Dreamers Pub Caffee,sports pub,"cozy,lively,local_favorite,sports_bar,loud,crowded,small_space,good_for_night_out,group_friendly,solo_friendly,outdoor_seating,cheap,good_value,friendly_staff"
826,0,Corks Cozy Winebar,wine bar,"cozy,local_favorite,specialty_coffee,wine"
1992,0,Silence Pub,locals' pub,"cozy,casual,local_favorite,lively,good_for_night_out,good_for_daytime,group_friendly,solo_friendly,drinks_focused,craft_beer,cheap,friendly_staff"
2419,0,YoYo Bar - Park IOR,park bar,"lively,casual,outdoor_seating,local_favorite,good_value,friendly_staff,drinks_focused,cocktails,good_for_daytime,good_for_night_out,group_friendly,dog_friendly"
43,1,5 To Go,coffee shop,"good_value,friendly_staff,fast_service,no_wait,good_for_daytime,solo_friendly,casual,good_coffee"
1595,1,NOMONYM Coffee Roastery,specialty coffee roastery,"specialty_coffee,good_coffee,cozy,small_space,local_favorite,hidden_gem,solo_friendly,good_for_daytime,friendly_staff,good_value,dog_friendly,easy_parking"
2312,1,"Tucano Coffee Zimbabwe - Victoriei, București",coffee shop,"cozy,stylish,good_for_daytime,specialty_coffee,good_coffee,desserts,vegetarian_options,laptop_friendly,study_friendly,solo_friendly,group_friendly,outdoor_seating,dog_friendly,good_value,inconsistent_service"
617,1,Cafenea Meraki,specialty coffee cafe,"specialty_coffee,good_coffee,tea,food_focused,quiet,cozy,casual,local_favorite,laptop_friendly,work_friendly,good_for_daytime,cheap"
1285,1,Koffeine Coșbuc,specialty coffee shop,"specialty_coffee,good_coffee,friendly_staff,cozy,small_space,quiet,good_for_daytime,cheap,good_value,fast_service,solo_friendly,outdoor_seating"
998,1,Fauna Coffee Shop,neighborhood coffee shop,"cozy,quiet,specialty_coffee,good_coffee,food_focused,laptop_friendly,solo_friendly,date_spot,local_favorite,good_for_daytime,good_value,small_space"


,recommendation_rank,profile_cluster,recommendation_score,similarity,tag_overlap,axis_similarity,quality_score,price_match,name,ai_place_type_summary,ai_card_summary,ai_tags_csv,google_rating,google_user_rating_count,map_visibility_score,based_on_favorites,place_id
0,26,0,0.890830,0.848296,0.619048,0.900625,0.880556,0.995,Boogie Bar,rock bar,"A cozy rock-and-blues bar with a spacious garden, self-service drinks, and a relaxed hangout feel for friends or small groups.","cozy,lively,casual,local_favorite,hidden_gem,group_friendly,solo_friendly,outdoor_seating,dog_friendly,good_for_daytime,good_for_night_out,craft_beer,drinks_focused,good_value,reservation_recommended",4.7,2076.0,87.6,Dreamers Pub Caffee; Corks Cozy Winebar; Silence Pub; YoYo Bar - Park IOR,ChIJzVzm_zP_sUARSNY5qOejefI
1,27,0,0.888984,0.872024,0.523810,0.944375,0.869634,0.875,QP Pub,casual pub and bar,"A cozy, music-loving pub with friendly service, late kitchen hours, and a strong mix of drinks, burgers, and hearty pub food.","cozy,local_favorite,lively,casual,outdoor_seating,smoking_area,good_for_daytime,good_for_night_out,group_friendly,dog_friendly,sports_bar,good_value",4.7,1476.0,86.3,Dreamers Pub Caffee; Corks Cozy Winebar; Silence Pub; YoYo Bar - Park IOR,ChIJE4yXZUj_sUARnaNfx0WjgvM
2,28,0,0.887733,0.898514,0.545455,0.905000,0.781040,0.755,Friends Pub,neighborhood pub and restaurant,"Lively neighborhood pub with a big terrace, sports-watching energy, and good cocktails, better for drinks and groups than for a standout meal.","lively,crowded,casual,local_favorite,outdoor_seating,good_for_night_out,good_for_daytime,group_friendly,sports_bar,cocktails,craft_beer,food_focused,good_value,inconsistent_service",4.4,1324.0,74.2,Dreamers Pub Caffee; Corks Cozy Winebar; Silence Pub; YoYo Bar - Park IOR,ChIJp3Ip58j-sUAR2qAEdf9UsXU
3,29,0,0.886417,0.857989,0.600000,0.853125,0.857733,0.975,Wicked Bar,rock bar,"A lively rock bar with vintage character, friendly staff, and a big drink selection, best for beers, music, and late-night hangs.","lively,casual,local_favorite,hidden_gem,good_for_night_out,good_for_daytime,group_friendly,solo_friendly,date_spot,outdoor_seating,smoking_area,cocktails,wine,craft_beer,good_value,friendly_staff,inconsistent_service,loud,crowded,dog_friendly",4.7,1075.0,84.8,Dreamers Pub Caffee; Corks Cozy Winebar; Silence Pub; YoYo Bar - Park IOR,ChIJnzptfVj_sUARbtnjvd5J9qE
4,30,0,0.883790,0.913066,0.450000,0.862500,0.761944,0.925,Family Pub,neighborhood pub,"A cozy neighborhood bar with a friendly, homey feel, good drinks, outdoor seating, and fair prices that make it easy to settle in.","cozy,local_favorite,outdoor_seating,good_for_daytime,good_for_night_out,sports_bar,good_value,friendly_staff,good_coffee,drinks_focused,reservation_recommended,easy_parking",4.9,62.0,71.5,Dreamers Pub Caffee; Corks Cozy Winebar; Silence Pub; YoYo Bar - Park IOR,ChIJ48eYXQD_sUARhSZ1Jkk_8Vo
5,31,0,0.881765,0.866107,0.454545,0.969375,0.824455,0.975,TwoBastardsPub,cozy pub,"Cozy, rock-leaning pub with standout beer, cocktails, and warm staff; best for a relaxed drink with friends or a casual night out.","cozy,lively,local_favorite,friendly_staff,good_value,drinks_focused,craft_beer,cocktails,wine,group_friendly,outdoor_seating,reservation_recommended,wheelchair_accessible",4.8,257.0,80.6,Dreamers Pub Caffee; Corks Cozy Winebar; Silence Pub; YoYo Bar - Park IOR,ChIJ2w0QhQ7_sUARSEblGzBV9zE
6,32,0,0.881270,0.855081,0.571429,0.923125,0.824259,0.825,Rainbow Coffee&more,casual cocktail bar,"A lively, budget-friendly bar with cheap cocktails, a roomy terrace, and a relaxed hangout feel, best for casual drinks with friends.","casual,lively,crowded,local_favorite,outdoor_seating,good_for_night_out,good_for_daytime,drinks_focused,cheap,good_value,cocktails,friendly_staff,inconsistent_service",4.6,915.0,80.3,Dreamers Pub Caffee; Corks Cozy Winebar; Silence Pub; YoYo Bar - Park IOR,ChIJL0QSNED_sUARwwSMUkxT-h4
7,33,0,0.879057,0.867063,0.550000,0.906250,0.746636,0.905,Lucky Point 

In [26]:
pd.read_csv('/Users/ilya/Documents/VisualStudioCode/SLOCO/data_scraping/output/backend_export/backend_dataset_20260531_223252/locations.csv')

place_id                           name   latitude  \
0     ChIJaX6OBQD_sUARpsOX3IMt2R4            "Latte" Coffe to go  44.420174   
1     ChIJ_xaqNQMCskAR8aQ8oHos1Ro              "Seneca Anticafe"  44.458479   
2     ChIJnUA9FgD_sUAR2ooJ3CWb0Zc       "TABERNA" LA ULTIMA Y YA  44.415583   
3     ChIJAQAckz__sUARPQnYMKtxIRQ                        #ALTFEL  44.431214   
4     ChIJbYXdJwD_sUAR2yjbAGOKJPM              #Altfel Park Lake  44.420245   
...                           ...                            ...        ...   
2503  ChIJYRGVLgD_sUARJXu-GmrIc3k                         Μαγαζί  44.432056   
2504  ChIJud_K0Mf_sUARCBOJ3CqSGy4                            Їжа  44.436600   
2505  ChIJKWe2SAD_sUARyR2nztHkGXI               בית קפה מרה מורה  44.441418   
2506  ChIJlVw_XWP_sUARSKlubPOvqZY  “ Caffe' del Moro “ Bucharest  44.428231   
2507  ChIJAe7KFAD_sUARLcNGiAmXP-g           🌈Rainbow Coffee&more  44.400635   

      longitude  \
0     26.065933   
1     26.078725   
2     26.078422   
3     26.099882   
4     26.150115   
...         ...   
2503  26.097499   
2504  26.098917   
2505  26.097890   
2506  26.071694   
2507  26.101239   

                                                                       formatted_address  \
0                                     Strada Mihail Sebastian, 051734 București, Romania   
1                                 Strada Arhitect Ion Mincu 1, 011365 București, Romania   
2     lote 20B entrada a cruz blanca san juan sacatepequez guatemala, București, Romania   
3                                           Strada Smârdan 29, 030076 București, Romania   
4                                              Str. Liviu Rebreanu 4, București, Romania   
...                                                                                  ...   
2503                                      Calea Victoriei 12A, 030026 București, Romania   
2504                                      Strada Academiei 21, 030167 București, Romania   
2505                               Strada Benjamin Franklin 5, 030167 București, Romania   
2506                      Strada Doctor Grigore Romniceanu 16, 050576 București, Romania   
2507                                     Șoseaua Olteniței 12, 020752 București, Romania   

                                                  details_shortFormattedAddress  \
0                                            Strada Mihail Sebastian, București   
1                                        Strada Arhitect Ion Mincu 1, București   
2     lote 20B entrada a cruz blanca san juan sacatepequez guatemala, București   
3                                                  Strada Smârdan 29, București   
4                                              Str. Liviu Rebreanu 4, București   
...                                                                         ...   
2503                                             Calea Victoriei 12A, București   
2504                                             Strada Academiei 21, București   
2505                                      Strada Benjamin Franklin 5, București   
2506                             Strada Doctor Grigore Romniceanu 16, București   
2507                                   Bloc 2D, Șoseaua Olteniței 12, București   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         